# 阶段 2 · 步骤 2：Dijkstra 求最短路径

用**固定权重**（不用随机数），方便手动验算。

Dijkstra 直观过程：
1. 维护表格：源点到每个节点的「当前已知最短距离」，初始全是 ∞，源点是 0
2. 每轮挑距离最小的未确定节点，标记为已确定
3. 看它的邻居：经过它中转会不会更近？会就更新
4. 重复直到全部确定

NetworkX 封装好了，你只管调用。逐 cell 往下走。

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

print('imports OK')

## 1. 建图：固定权重

`add_weighted_edges_from` 一次加「边 + 权重」，格式 `(u, v, w)`。

先猜一下：0 到 3 走哪条路最近？是跳数少的 `0-1-3` 吗？跑完再对答案。

In [ ]:
G = nx.Graph()
G.add_weighted_edges_from([
    (0, 1, 0.7),
    (1, 2, 0.9),
    (2, 3, 0.4),
    (3, 4, 0.2),
    (0, 4, 0.8),
    (1, 3, 0.5),
])
print(f"节点: {G.number_of_nodes()}, 边: {G.number_of_edges()}")

## 2. 最短路径与最短距离

- `shortest_path` 返回**节点序列**（走哪条路）
- `shortest_path_length` 返回**累计权重**（要走多远，一个数字）

注意 `weight='weight'` 参数——不写的话 NetworkX 默认按跳数算（BFS），就失去权重信息了。

In [ ]:
path = nx.shortest_path(G, source=0, target=3, weight='weight')
dist = nx.shortest_path_length(G, source=0, target=3, weight='weight')

print(f"最短路径 0 -> 3: {path}")
print(f"最短距离: {dist}")

## 3. 手动验证（重要习惯）

「路径长度 = 沿途权重之和」。以后生成训练数据时，靠这个习惯确认标签没打错。

`zip(path[:-1], path[1:])` 把节点序列变成相邻节点对——这是「路径转边」的标准写法，阶段 2 打标签全靠它。

In [ ]:
manual = sum(G[u][v]['weight'] for u, v in zip(path[:-1], path[1:]))
print(f"手动累加: {manual} (应等于 {dist})")
print(f"路径转边对: {list(zip(path[:-1], path[1:]))}")

## 4. 单源到全部节点的距离

一次算出源点 0 到**所有**节点的最短距离。这个字典是后面给节点造特征的材料（BFS 跳数版本同理）。

In [ ]:
print(f"0 到各节点的距离: {nx.single_source_dijkstra_path_length(G, 0, weight='weight')}")

## 5. 可视化：最短路径标红

红线 = Dijkstra 找到的路。对照权重，确认它确实是权重和最小的。

In [ ]:
pos = nx.spring_layout(G, seed=42)
path_edges = list(zip(path[:-1], path[1:]))
edge_colors = ['red' if (u, v) in path_edges or (v, u) in path_edges else 'gray'
               for u, v in G.edges()]

nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=700,
        font_weight='bold', edge_color=edge_colors, width=2)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'))
plt.show()

## 检查点

1. 最短路径是 `0-1-3`（2 跳）还是 `0-4-3`（2 跳）？为什么距离不同？
2. 如果调用 `nx.shortest_path(G, 0, 3)`（不写 `weight` 参数），结果会变成什么？动手试一下。
3. `zip(path[:-1], path[1:])` 对 `[0, 4, 3]` 输出什么？

答对后进入 `02_data_generation.ipynb`（步骤 3，核心）。